In [1]:
import time
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder ,MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Create output directory for deliverables

In [2]:
csv_path = "processed_data/procesed_customer_churn.csv" 
df = pd.read_csv(csv_path)

In [3]:
print(f"Dataset Shape: {df.shape}")

# TODO: Change 'Churn' if your target column has a different name
target_col = 'Churn' 

# Classify features based on raw variable types
num_features = df.drop(columns=[target_col]).select_dtypes(include=[np.number]).columns.tolist()
cat_features = df.drop(columns=[target_col]).select_dtypes(exclude=[np.number]).columns.tolist()

X = df.drop(columns=[target_col])
y = df[target_col]


Dataset Shape: (440832, 13)


In [4]:
# ==========================================
# 2. DATA SPLITTING & STRATIFICATION
# ==========================================
# Train (70%), Validation (15%), Test (15%) splits with stratification for imbalance
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.1765, random_state=42, stratify=y_train_val
)

# ==========================================
# 3. PREPROCESSING PIPELINE
# ==========================================
num_transformer = Pipeline(steps=[('scaler', MinMaxScaler())])
cat_transformer = Pipeline(steps=[('encoder', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

# Processed splits for standard models
X_train_scaled = preprocessor.fit_transform(X_train)
X_val_scaled = preprocessor.transform(X_val)
X_test_scaled = preprocessor.transform(X_test)

# Unscaled pipeline variant specifically for the required SVM scaling experiment
preprocessor_unscaled = ColumnTransformer(transformers=[
    ('cat', cat_transformer, cat_features)
], remainder='passthrough') 

X_train_unscaled = preprocessor_unscaled.fit_transform(X_train)
X_val_unscaled = preprocessor_unscaled.transform(X_val)
X_test_unscaled = preprocessor_unscaled.transform(X_test)

In [5]:
# ==========================================
# 4. MODEL DICTIONARY SETUP (Required Benchmarks)
# ==========================================
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
#from xgboost import XGBClassifier
from sklearn.svm import LinearSVC

models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    
    # EXPERIMENT 1: Compare at least two values of K for KNN
    "KNN (K=3) [Low K]": KNeighborsClassifier(n_neighbors=3),
    "KNN (K=15) [High K]": KNeighborsClassifier(n_neighbors=15),
    
    # EXPERIMENT 2 & 5: Underfitting (Shallow) vs Overfitting (Deep) Decision Tree
    "Decision Tree (Shallow, Depth=3) [Underfit Model]": DecisionTreeClassifier(max_depth=3, random_state=42),
    "Decision Tree (Deep/Max, Depth=20) [Overfit Model]": DecisionTreeClassifier(max_depth=20, random_state=42),
    
    # EXPERIMENT 3: Compare Decision Tree with Random Forest
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    
    # EXPERIMENT 4: Compare Random Forest with a Boosting Model
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    #"XGBoost": XGBClassifier(random_state=42, eval_metric='logloss'),
    
    # EXPERIMENT 6: Compare SVM with and without appropriate feature scaling
    "SVM (Scaled)": LinearSVC(random_state=42),
    "SVM (Unscaled)": LinearSVC(random_state=42)
}

In [6]:
# ==========================================
# 5. BENCHMARK EXECUTION ENGINE
# ==========================================
results_list = []

print("\n=== Step 2: Running Benchmarks ===")
for name, model in models.items():
    # Route to unscaled data if executing the specific unscaled SVM baseline
    X_tr = X_train_unscaled if "Unscaled" in name else X_train_scaled
    X_v = X_val_unscaled if "Unscaled" in name else X_val_scaled
    X_te = X_test_unscaled if "Unscaled" in name else X_test_scaled
    
    # Track Training Time
    start_train = time.time()
    model.fit(X_tr, y_train)
    train_time = time.time() - start_train
    
    # Track Inference Time 
    start_infer = time.time()
    val_preds = model.predict(X_v)
    infer_time = time.time() - start_infer
    
    # Extract probabilities
    val_probs = model.predict_proba(X_v)[:, 1] if hasattr(model, "predict_proba") else val_preds
    train_preds = model.predict(X_tr)
    test_preds = model.predict(X_te)
    
    metrics = {
        "Model / Experiment Configuration": name,
        "Train Acc": accuracy_score(y_train, train_preds),
        "Val Acc": accuracy_score(y_val, val_preds),
        "Val Precision": precision_score(y_val, val_preds, zero_division=0),
        "Val Recall": recall_score(y_val, val_preds),
        "Val F1": f1_score(y_val, val_preds),
        "Val ROC-AUC": roc_auc_score(y_val, val_probs),
        "Test Acc": accuracy_score(y_test, test_preds),
        "Train Time (s)": train_time,
        "Inference Time (s)": infer_time
    }
    results_list.append(metrics)

    print(f"=========Model :{name} done ==============")



=== Step 2: Running Benchmarks ===
=========Model :Logistic Regression done ==============
=========Model :KNN (K=3) [Low K] done ==============
=========Model :KNN (K=15) [High K] done ==============
=========Model :Decision Tree (Shallow, Depth=3) [Underfit Model] done ==============
=========Model :Decision Tree (Deep/Max, Depth=20) [Overfit Model] done ==============
=========Model :Random Forest done ==============
=========Model :Gradient Boosting done ==============
=========Model :SVM (Scaled) done ==============
=========Model :SVM (Unscaled) done ==============


In [7]:
# Create output directory for deliverables
output_dir = "benchmark_outputs"
os.makedirs(output_dir, exist_ok=True)

In [8]:
# ==========================================
# 6. EXPORT DELIVERABLES (CSV & PLOTS)
# ==========================================
results_df = pd.DataFrame(results_list)
csv_output_path = os.path.join(output_dir, "model_benchmark_results.csv")
results_df.to_csv(csv_output_path, index=False)
print(f"\n[SUCCESS] Saved clean results table to: {csv_output_path}")
print(results_df.to_string(index=False))

# Plot and save confusion matrices 
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Subplot 1: Overfitting Deep Tree
cm_overfit = confusion_matrix(y_val, models["Decision Tree (Deep/Max, Depth=20) [Overfit Model]"].predict(X_val_scaled))
sns.heatmap(cm_overfit, annot=True, fmt='d', cmap='Oranges', ax=axes[0],
            xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
axes[0].set_title('Overfit Model: Deep Tree (Depth=20)')
axes[0].set_ylabel('Actual Label')
axes[0].set_xlabel('Predicted Label')


plt.tight_layout()
plot_output_path = os.path.join(output_dir, "experiment_confusion_matrices.png")
plt.savefig(plot_output_path, dpi=300)
plt.close()
print(f"[SUCCESS] Saved final evaluation plots to: {plot_output_path}\n")


[SUCCESS] Saved clean results table to: benchmark_outputs/model_benchmark_results.csv
                  Model / Experiment Configuration  Train Acc  Val Acc  Val Precision  Val Recall   Val F1  Val ROC-AUC  Test Acc  Train Time (s)  Inference Time (s)
                               Logistic Regression   0.980627 0.980011       0.984598    0.980083 0.982336     0.996616  0.980854        0.463105            0.002866
                                 KNN (K=3) [Low K]   0.993771 0.987858       0.992433    0.986109 0.989261     0.995557  0.988416        0.015927           37.328897
                               KNN (K=15) [High K]   0.988045 0.985908       0.991745    0.983336 0.987522     0.999065  0.986873        0.019387           38.638309
 Decision Tree (Shallow, Depth=3) [Underfit Model]   0.991512 0.992183       0.986403    1.000000 0.993155     0.998219  0.991577        0.682909            0.007334
Decision Tree (Deep/Max, Depth=20) [Overfit Model]   1.000000 0.999803       0.9998